# Building a Question Classifier using Tf-Idf Vectorization and Artificial Neural Networks

We have used a question classification dataset. We will try and classify questions based on their text into one of the following six classes:

* ABBREVIATION 
* ENTITY 
* DESCRIPTION 
* HUMAN 
* LOCATION 
* NUMERIC

In [1]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer 
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/shekhabdullahayubi/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/shekhabdullahayubi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Reading Datasets

In [2]:
train_data = open('Dataset/training_data.txt', 'r+')
test_data = open('Dataset/test_dataset.txt', 'r+')

train = pd.DataFrame(train_data.readlines(), columns = ['Question'])
test = pd.DataFrame(test_data.readlines(), columns = ['Question'])

In [4]:
train.head()

,Question
0,DESC:manner How did serfdom develop in and the...
1,ENTY:cremat What films featured the character ...
2,DESC:manner How can I find a list of celebriti...
3,ENTY:animal What fowl grabs the spotlight afte...
4,ABBR:exp What is the full form of .com ?\n


### Splitting data points to obtain Question string and Coarse and Fine question categories

In [5]:
train['QType'] = train.Question.apply(lambda x: x.split(' ', 1)[0])
train['Question'] = train.Question.apply(lambda x: x.split(' ', 1)[1])
train['QType-Coarse'] = train.QType.apply(lambda x: x.split(':')[0])
train['QType-Fine'] = train.QType.apply(lambda x: x.split(':')[1])
test['QType'] = test.Question.apply(lambda x: x.split(' ', 1)[0])
test['Question'] = test.Question.apply(lambda x: x.split(' ', 1)[1])
test['QType-Coarse'] = test.QType.apply(lambda x: x.split(':')[0])
test['QType-Fine'] = test.QType.apply(lambda x: x.split(':')[1])

In [7]:
train.tail()

,Question,QType,QType-Coarse,QType-Fine
5447,What 's the shape of a camel 's spine ?\n,ENTY:other,ENTY,other
5448,What type of currency is used in China ?\n,ENTY:currency,ENTY,currency
5449,What is the temperature today ?\n,NUM:temp,NUM,temp
5450,What is the temperature for cooking ?\n,NUM:temp,NUM,temp
5451,What currency is used in Australia ?,ENTY:currency,ENTY,currency


### Popping out QType and QType-Fine variables as we would focus on Coarse Category Classification

In [8]:
train.pop('QType')
train.pop('QType-Fine')
test.pop('QType')
test.pop('QType-Fine')

0           dist
1           city
2           desc
3            def
4           date
         ...    
495          ind
496     currency
497        count
498    substance
499          def
Name: QType-Fine, Length: 500, dtype: object

### Lets look at classes involved in the our dataset

In [9]:
classes = np.unique(np.array(train['QType-Coarse']))
classes

array(['ABBR', 'DESC', 'ENTY', 'HUM', 'LOC', 'NUM'], dtype=object)

### Let's Label Encode our classes to convert them into integer identifiers

In [10]:
le = LabelEncoder()
le.fit(pd.Series(train['QType-Coarse'].tolist() + test['QType-Coarse'].tolist()).values)
train['QType-Coarse'] = le.transform(train['QType-Coarse'].values)
test['QType-Coarse'] = le.transform(test['QType-Coarse'].values)

### Preprocess our Dataset

In [11]:
all_corpus = pd.Series(train.Question.tolist() + test.Question.tolist()).astype(str)

In [21]:
def text_clean(corpus):
    '''
    Purpose : Function to keep only alphabets, digits and certain words (punctuations, qmarks, tabs etc. removed)
    
    Input : Takes a text corpus, 'corpus' to be cleaned
    
    Output : Returns the cleaned text corpus
    '''
    def clean_row(row):
        text = re.sub(r'[^a-zA-Z0-9]', ' ', str(row))
        return ' '.join(text.lower().split())

    if isinstance(corpus, pd.Series):
        return corpus.astype(str).apply(clean_row)
    return pd.Series([clean_row(row) for row in corpus])


In [14]:
def stopwords_removal(corpus):
    wh_words = ['who', 'what', 'when', 'why', 'how', 'which', 'where', 'whom']
    stop = set(stopwords.words('english'))
    for word in wh_words:
        stop.remove(word)
    corpus = [[x for x in x.split() if x not in stop] for x in corpus]
    return corpus

In [15]:
def lemmatize(corpus):
    lem = WordNetLemmatizer()
    corpus = [[lem.lemmatize(x, pos = 'v') for x in x] for x in corpus]
    return corpus

In [16]:
def stem(corpus, stem_type = None):
    if stem_type == 'snowball':
        stemmer = SnowballStemmer(language = 'english')
        corpus = [[stemmer.stem(x) for x in x] for x in corpus]
    else :
        stemmer = PorterStemmer()
        corpus = [[stemmer.stem(x) for x in x] for x in corpus]
    return corpus

In [17]:
def preprocess(corpus, cleaning = True, stemming = False, stem_type = None, lemmatization = False, remove_stopwords = True):
    
    '''
    Purpose : Function to perform all pre-processing tasks (cleaning, stemming, lemmatization, stopwords removal etc.)
    
    Input : 
    'corpus' - Text corpus on which pre-processing tasks will be performed
    
    'cleaning', 'stemming', 'lemmatization', 'remove_stopwords' - Boolean variables indicating whether a particular task should 
                                                                  be performed or not
    'stem_type' - Choose between Porter stemmer or Snowball(Porter2) stemmer. Default is "None", which corresponds to Porter
                  Stemmer. 'snowball' corresponds to Snowball Stemmer
    
    Note : Either stemming or lemmatization should be used. There's no benefit of using both of them together
    
    Output : Returns the processed text corpus
    
    '''
    if cleaning == True:
        corpus = text_clean(corpus)
    
    if remove_stopwords == True:
        corpus = stopwords_removal(corpus)
    else :
        corpus = [[x for x in x.split()] for x in corpus]
    
    if lemmatization == True:
        corpus = lemmatize(corpus)
        
        
    if stemming == True:
        corpus = stem(corpus, stem_type)
    
    corpus = [' '.join(x) for x in corpus]
        

    return corpus

In [22]:
all_corpus = preprocess(all_corpus, remove_stopwords = True)

In [26]:
all_corpus[:5]

['how serfdom develop leave russia',
 'what films featured character popeye doyle',
 'how find list celebrities real names',
 'what fowl grabs spotlight chinese year monkey',
 'what full form com']

In [27]:
train_corpus = all_corpus[0:train.shape[0]]
test_corpus = all_corpus[train.shape[0]:]

### TF-IDF based Vectorization of text 

In [31]:
vectorizer = TfidfVectorizer()
tf_idf_matrix_train = vectorizer.fit_transform(train_corpus)

In [35]:
tf_idf_matrix_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 30578 stored elements and shape (5452, 8294)>

In [36]:
tf_idf_matrix_test = vectorizer.transform(test_corpus)
tf_idf_matrix_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1688 stored elements and shape (500, 8294)>

### We would use Keras for building our network architecture

In [40]:
import keras 
from keras.models import Sequential, Model 
from keras import layers
from keras.layers import Dense, Dropout, Input
from keras.utils import to_categorical

### Converting the Class labels into one hot encoded vectors

In [41]:
y_train = to_categorical(train['QType-Coarse'], train['QType-Coarse'].nunique())
y_test = to_categorical(test['QType-Coarse'], train['QType-Coarse'].nunique())

In [45]:
tf_idf_matrix_train.shape[1]

8294

### Defining and Building our network architecture

In [50]:
model = Sequential()

model.add(Dense(128, activation='relu', input_dim=tf_idf_matrix_train.shape[1])) # The Hidden Processing Layer
model.add(Dropout(0.3)) # The Overfitting Guard
# The Output Layer. 6 corresponds to the number of classes in the target variable, 
# 'QType-Coarse'. Softmax is used as the activation function since this is a multi-class classification problem
model.add(Dense(6, activation='softmax')) 
# Compiling the model with Adam optimizer and Categorical Crossentropy loss function. 
# Categorical Accuracy is used as the evaluation metric since this is a multi-class classification problem
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['categorical_accuracy']) 
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 128)            │     1,061,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,062,534 (4.05 MB)

 Trainable params: 1,062,534 (4.05 MB)

 Non-trainable params: 0 (0.00 B)

`activation='softmax':` Takes the raw numbers from the `6 nodes` and squashes them into a probability distribution that adds up to exactly 1.0 (or 100%). For example, it will output something like: `[0.05, 0.80, 0.02, 0.03, 0.05, 0.05]`, telling you there is an `80%` chance the text belongs to `category #2`.

### **Introduction to Adam**

"`Adam`" stands for **Adaptive Moment Estimation**, the Adam Optimizer is like giving you a pair of high-tech, self-adjusting rocket boots to get down that mountain.

Adam behaves across different AI fields:

1. Natural Language Processing (NLP) & LLMs
* The Scenario: Text data relies heavily on word embeddings. Common words (like "the", "is") appear constantly, while rare vocabulary words (like "photosynthesis") appear once in a million tokens.
* Adam's Behavior: For common words, Adam dials down the step size so they don’t overwhelm the model. For rare words, Adam notices they seldom get updated and amplifies their step size when they do appear. This ensures the model actually learns from rare context.

2. Computer Vision (Convolutional Neural Networks)
* The Scenario: Images contain vast landscapes of flat pixels (like a clear blue sky) punctuated by sudden, sharp edges (like the outline of a building).
* Adam's Behavior: When traversing flat regions of an image, Adam builds momentum to slide across the uninformative pixels quickly. When it encounters sharp, high-contrast boundaries, its internal braking system kicks in, slowing down the updates to map out the fine textures without destroying the weights.

3. Deep Reinforcement Learning (RL)
* The Scenario: An AI agent interacts with an environment (like a video game). The feedback loop is chaotic; the agent receives no rewards for a long time, followed by a sudden spike of rewards.
* Adam's Behavior: Because the reward signals are highly volatile, Adam stabilizes the policy updates, preventing the AI from completely forgetting good older strategies when it hits a string of bad luck.

### Model training

In [51]:
training_history = model.fit(tf_idf_matrix_train, y_train, epochs=10, batch_size=100)

Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - categorical_accuracy: 0.3434 - loss: 1.7023
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - categorical_accuracy: 0.7782 - loss: 1.3466
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - categorical_accuracy: 0.8877 - loss: 0.8887
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - categorical_accuracy: 0.9332 - loss: 0.5479
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - categorical_accuracy: 0.9582 - loss: 0.3502
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - categorical_accuracy: 0.9741 - loss: 0.2386
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - categorical_accuracy: 0.9829 - loss: 0.1688
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - categorical_accuracy: 0.9870 - loss: 0.1265
Epoch 9/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - categorical_accuracy: 0.9899 - loss: 0.0971
Epoch 10/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - categorical_accuracy: 0.9932 - loss: 0.0756


### Model Evaluation on test data based on Accuracy

In [49]:
loss, accuracy = model.evaluate(tf_idf_matrix_test, y_test, verbose=False)
print("Testing Accuracy:  {:.4f}".format(accuracy))

Testing Accuracy:  0.8520


### Save the model architecture and weights 

In [56]:
import h5py
model_structure = model.to_json()
with open("Dataset/saved_model/question_classification_model.json", "w") as json_file:
    json_file.write(model_structure)
model.save_weights("Dataset/saved_model/question_classification.weights.h5")